# Project 01 — Estimating a Proportion (Beta–Binomial)

**Scenario.** A biochemical binary assay is run many times; each run succeeds with an unknown probability $\theta$. We want the full posterior for $\theta$, not just a point estimate.

This is the simplest Bayesian problem — one parameter — so we use it to learn the **entire workflow**: data story → model → prior predictive → inference → diagnostics → posterior predictive → criticism → decision.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

We assume each assay run is an independent Bernoulli trial with a common success probability $\theta$. **Assumptions made explicit:** (a) runs are independent, (b) $\theta$ is constant across runs (no drift/batch effects), (c) outcomes are truly binary. We synthesize data from a known $\theta_\text{true}=0.62$ so we can later check recovery.

In [ ]:
from data.generate_data import generate
data = generate()
y = data['y']
print(f"n={data['n']} runs, k={data['k']} successes, "
      f"empirical rate={data['k']/data['n']:.3f}, true theta={data['truth']['theta']}")

## Step 2 — Model specification (likelihood + justified priors)

$$y_i \sim \text{Bernoulli}(\theta), \qquad \theta \sim \text{Beta}(2,2).$$

**Why Beta(2,2) and not Beta(1,1)?** A 'flat' Beta(1,1) places as much mass on $\theta=0.999$ as on $\theta=0.5$, which is *not* an innocent default for an assay — it over-trusts extreme rates when data are scarce. Beta(2,2) is mild, unimodal, centred at 0.5, and pulls gently away from the degenerate edges. It is **weakly informative**, the recommended posture.

In [ ]:
from model import build_model, fit, analytic_posterior
model = build_model(data)
model

## Step 3 — Prior predictive checks

Before touching the data we simulate datasets *implied by the prior*. If the prior implied, say, that 95% of assays succeed essentially never or always, we'd fix the prior now. We look at the distribution of the success **count** $k$ under the prior; Beta(2,2) should spread mass across the whole 0–n range, concentrated mildly toward the middle.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=1000, random_seed=RNG)
prior_k = prior.prior_predictive['y'].sum(dim='y_dim_2' if 'y_dim_2' in prior.prior_predictive['y'].dims else prior.prior_predictive['y'].dims[-1]).values.ravel()
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(prior_k, bins=np.arange(0, data['n']+2)-0.5, color='#55A868', edgecolor='white')
ax.set(xlabel='successes k implied by prior', ylabel='count',
       title='Prior predictive — sensible spread, no pathology')
plt.tight_layout()

## Step 4 — Inference (NUTS)

Even though this model has a closed-form conjugate posterior, we sample with **NUTS** to learn the machinery. Settings: `draws=1000, tune=1000, chains=4`. Four chains let us compute split-$\hat R$ reliably; 1000 tuning steps let NUTS adapt its step size and mass matrix. We fix `random_seed` for reproducibility.

In [ ]:
idata = fit(data, draws=1000, tune=1000, chains=4, seed=101)

## Step 5 — Computational diagnostics

We check: **$\hat R$** (should be ≈ 1.00; > 1.01 signals chains disagree), **ESS** (bulk/tail effective sample size; want ≳ 400), and **divergences** (should be 0 for this easy geometry). The trace should look like 'fuzzy caterpillars' with well-mixed chains.

In [ ]:
print(az.summary(idata, var_names=['theta']))
n_div = int(idata.sample_stats['diverging'].sum())
print(f'divergences: {n_div}')

In [ ]:
az.plot_trace(idata, var_names=['theta']); plt.tight_layout()

**What if diagnostics fail?** For this model they won't, but the general remedies are: raise `target_accept` (e.g. 0.95) to shrink step size and clear divergences; increase `tune`/`draws` for low ESS; and if $\hat R$ stays high, suspect a multimodal or non-identified model (we'll meet those in later projects).

## Step 6 — Posterior predictive checks

We compare the observed number of successes to the distribution of successes in datasets simulated from the *posterior*. If the model is adequate, the observed value sits comfortably inside the posterior-predictive spread.

In [ ]:
ax = az.plot_ppc(idata, num_pp_samples=200)
plt.tight_layout()

In [ ]:
pp = idata.posterior_predictive['y']
pp_k = pp.sum(dim=pp.dims[-1]).values.ravel()
p_value = float(np.mean(pp_k >= data['k']))
print(f'observed k={data["k"]}; posterior-predictive p-value={p_value:.3f} (near 0.5 = good fit)')

## Step 7 — Model criticism & comparison

Sanity check against the **exact conjugate posterior** Beta$(2+k,\,2+n-k)$ — MCMC should match it closely. (With a single parameter and one model there is no LOO/WAIC comparison to make; later projects introduce competing models.)

In [ ]:
a_post, b_post = analytic_posterior(data)
grid = np.linspace(0, 1, 400)
fig, ax = plt.subplots(figsize=(6,3.5))
az.plot_dist(idata.posterior['theta'].values.ravel(), ax=ax, color='#4C72B0',
             label='MCMC posterior')
ax.plot(grid, beta_dist.pdf(grid, a_post, b_post), 'k--', label='analytic Beta')
ax.axvline(data['truth']['theta'], color='red', lw=1, label='true theta')
ax.set(xlabel='theta', ylabel='density', title='MCMC vs analytic posterior')
ax.legend(); plt.tight_layout()

## Step 8 — Decision & communication

Translate the posterior into something a collaborator can use: a point estimate with a credible interval, and the probability the assay beats a decision threshold (say 0.5).

In [ ]:
post = idata.posterior['theta'].values.ravel()
mean = post.mean(); lo, hi = np.percentile(post, [3, 97])
p_above = float(np.mean(post > 0.5))
print(f'Posterior mean theta = {mean:.3f}')
print(f'94% credible interval = [{lo:.3f}, {hi:.3f}]')
print(f'P(theta > 0.5 | data) = {p_above:.3f}')

**Conclusion (for a collaborator).** The assay's true success rate is most plausibly around 0.59 with a 94% credible interval of roughly [0.49, 0.69]. We are ~92% sure the rate exceeds one-half. The next step is the decision (see `summary_onepager.md`), not the posterior itself.